# Notebook 04 — MLP

Multi-Layer Perceptron — default configuration, Optuna tuning, final model.

**Prerequisite:** run `Notebook00_Setup_PCA300.ipynb` first.

Fixes vs. previous version: (1) models are fitted *in place* so the pickled `model_MLP*.pkl` files are the trained models (previously the unfitted object was saved); (2) 95% bootstrap CIs, per-class report and confusion matrix, identical to the other classifier notebooks; (3) Optuna tuning on the validation split by default; (4) final model reads its parameters from `optuna_result_MLP.json` instead of hard-coding them; (5) dataset guard against stale v2 runs.

## Step 0 — Setup

In [1]:
import os, warnings, pickle, json
import numpy as np
import pandas as pd
from pathlib import Path

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix)

warnings.filterwarnings("ignore")
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

DATASET_ROOT = "/home/ali/Desktop/Algerian_Lamiaceae_v3"
OUTPUT_DIR   = os.path.join(DATASET_ROOT, "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output: {OUTPUT_DIR}")


Device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Output: /home/ali/Desktop/Algerian_Lamiaceae_v3/results


## Step 1 — Load Cached Features and Train/Val/Test Split

In [2]:
HC_CACHE    = os.path.join(OUTPUT_DIR, "hc_features_cache.npz")
DEEP_CACHE  = os.path.join(OUTPUT_DIR, "deep_features_cache.npz")
SPLIT_CACHE = os.path.join(OUTPUT_DIR, "train_test_split.npz")

X_hc = np.load(HC_CACHE, allow_pickle=True)["X"]
print(f"Handcrafted features: {X_hc.shape}")

if os.path.exists(DEEP_CACHE):
    X_deep = np.load(DEEP_CACHE, allow_pickle=True)["X"]
    X_all  = np.concatenate([X_deep, X_hc], axis=1)
    print(f"Deep features: {X_deep.shape}")
else:
    X_all = X_hc
    print("Deep features not found -- using handcrafted only")
print(f"Fused features: {X_all.shape}")

split = np.load(SPLIT_CACHE, allow_pickle=True)
X_train_sc = split["X_train_sc"]
X_val_sc   = split["X_val_sc"]
X_test_sc  = split["X_test_sc"]
y_train_sm = split["y_train_sm"]
y_val_raw  = split["y_val_raw"]
y_test_raw = split["y_test_raw"]
n_components = int(split["n_components"])

with open(os.path.join(OUTPUT_DIR, "label_encoder_species.pkl"), "rb") as f:
    le = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "sc1.pkl"), "rb") as f:
    sc1 = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "sc2.pkl"), "rb") as f:
    sc2 = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "pca.pkl"), "rb") as f:
    pca = pickle.load(f)

class_names = list(le.classes_)
NUM_CLASSES = len(class_names)

print(f"Train: {X_train_sc.shape}  Val: {X_val_sc.shape}  Test: {X_test_sc.shape}")
print(f"Classes: {NUM_CLASSES} -- {class_names}")

test_support = pd.Series(y_test_raw).value_counts().sort_index()
print("\nTest-set support per class:")
for i, cls in enumerate(class_names):
    n = int(test_support.get(i, 0))
    flag = "  \u26a0 SMALL SUPPORT" if n < 30 else ""
    print(f"  {cls:<28} n={n:>5}{flag}")


Handcrafted features: (19145, 73)
Deep features: (19145, 1536)
Fused features: (19145, 1609)
Train: (35000, 300)  Val: (2372, 300)  Test: (2366, 300)
Classes: 7 -- [np.str_('Lamium_pooled'), np.str_('Lavandula_angustifolia'), np.str_('Lavandula_pooled'), np.str_('Leucas_aspera'), np.str_('Mentha_pooled'), np.str_('Ocimum_pooled'), np.str_('Plectranthus_pooled')]

Test-set support per class:
  Lamium_pooled                n= 1529
  Lavandula_angustifolia       n=  342
  Lavandula_pooled             n=  340
  Leucas_aspera                n=   11  ⚠ SMALL SUPPORT
  Mentha_pooled                n=   48
  Ocimum_pooled                n=   51
  Plectranthus_pooled          n=   45


In [3]:
# ── Guard: make sure this run uses the final v3 dataset (prevents stale v2 runs) ──
assert "Algerian_Lamiaceae_v3" in OUTPUT_DIR, f"Wrong OUTPUT_DIR: {OUTPUT_DIR}"
assert "Lamium_pooled" in class_names, f"Old class names detected: {class_names}"
print(f"Dataset check passed: v3 results folder, {len(class_names)} classes, "
      f"test n={X_test_sc.shape[0]:,}, val n={X_val_sc.shape[0]:,}")

Dataset check passed: v3 results folder, 7 classes, test n=2,366, val n=2,372


## Step 2 — Evaluation Utilities (identical to the other classifier notebooks)

In [4]:
N_BOOTSTRAP = 1000
SMALL_SUPPORT_THRESHOLD = 30

def evaluate(clf, Xtr, Xte, ytr, yte):
    """Fits clf IN PLACE and returns it alongside its metrics -- the same
    object is what gets pickled downstream, closing the unfitted-model bug."""
    clf.fit(Xtr, ytr)
    yp = clf.predict(Xte)
    yproba = None
    auc_s = None
    if hasattr(clf, "predict_proba"):
        try:
            yproba = clf.predict_proba(Xte)
            auc_s = round(roc_auc_score(yte, yproba, multi_class="ovr",
                                         average="macro") * 100, 2)
        except Exception:
            pass
    return {
        "Accuracy":  round(accuracy_score(yte, yp) * 100, 2),
        "Precision": round(precision_score(yte, yp, average="weighted", zero_division=0) * 100, 2),
        "Recall":    round(recall_score(yte, yp, average="weighted", zero_division=0) * 100, 2),
        "F1":        round(f1_score(yte, yp, average="weighted", zero_division=0) * 100, 2),
        "AUC": auc_s,
        "_yp": yp, "_yproba": yproba, "_fitted_clf": clf,
    }

def bootstrap_ci(y_true, y_pred, y_proba=None, n_boot=N_BOOTSTRAP, seed=GLOBAL_SEED):
    """Percentile bootstrap 95% CI for Accuracy/Precision/Recall/F1/AUC.
    Resamples where a bootstrap draw happens to omit a class are skipped
    for the AUC calculation only (multiclass OvR AUC requires every class
    present); this is tracked and reported, not silently ignored."""
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    n = len(y_true)
    accs, precs, recs, f1s, aucs = [], [], [], [], []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, yp = y_true[idx], y_pred[idx]
        accs.append(accuracy_score(yt, yp))
        precs.append(precision_score(yt, yp, average="weighted", zero_division=0))
        recs.append(recall_score(yt, yp, average="weighted", zero_division=0))
        f1s.append(f1_score(yt, yp, average="weighted", zero_division=0))
        if y_proba is not None:
            try:
                aucs.append(roc_auc_score(yt, y_proba[idx], multi_class="ovr", average="macro"))
            except ValueError:
                pass  # a class was missing from this particular resample

    def summarize(arr):
        arr = np.array(arr) * 100
        return {"mean": round(float(arr.mean()), 2),
                "ci_lo": round(float(np.percentile(arr, 2.5)), 2),
                "ci_hi": round(float(np.percentile(arr, 97.5)), 2)}

    result = {"Accuracy": summarize(accs), "Precision": summarize(precs),
              "Recall": summarize(recs), "F1": summarize(f1s)}
    if y_proba is not None:
        result["AUC"] = summarize(aucs) if aucs else None
        result["AUC_valid_resamples"] = len(aucs)
        result["AUC_skipped_resamples"] = n_boot - len(aucs)
    return result

def per_class_report(y_true, y_pred, class_names, small_support_threshold=SMALL_SUPPORT_THRESHOLD):
    report = classification_report(y_true, y_pred, labels=list(range(len(class_names))),
                                    target_names=class_names, output_dict=True, zero_division=0)
    rows = []
    for cls in class_names:
        r = report[cls]
        rows.append({"Class": cls, "Precision": round(r["precision"] * 100, 2),
                     "Recall": round(r["recall"] * 100, 2),
                     "F1": round(r["f1-score"] * 100, 2), "Support": int(r["support"])})
    df = pd.DataFrame(rows)
    small = df[df["Support"] < small_support_threshold]
    if not small.empty:
        print(f"  \u26a0 Small test support (<{small_support_threshold} samples) -- "
              f"per-class scores below should not be over-interpreted (comment 7):")
        for _, r in small.iterrows():
            print(f"     {r['Class']:<28} support={r['Support']:>4}  "
                  f"P={r['Precision']:.1f}%  R={r['Recall']:.1f}%  F1={r['F1']:.1f}%")
    return df

def save_full_result(name, m, y_true, class_names):
    """Fix: takes the fitted clf FROM m (the object evaluate() actually
    trained), not a separately-passed, possibly-unfitted reference."""
    clf = m["_fitted_clf"]
    yp, yproba = m["_yp"], m["_yproba"]

    ci = bootstrap_ci(y_true, yp, yproba)
    per_class = per_class_report(y_true, yp, class_names)
    cm = confusion_matrix(y_true, yp, labels=list(range(len(class_names))))

    result = {k: v for k, v in m.items() if not k.startswith("_")}
    result["Classifier"] = name
    result["Bootstrap_CI_95"] = ci
    result["N_bootstrap"] = N_BOOTSTRAP
    result["PerClass"] = per_class.to_dict(orient="records")

    out_path = os.path.join(OUTPUT_DIR, f"result_{name.replace(' ','_')}.json")
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2)

    mdl_path = os.path.join(OUTPUT_DIR, f"model_{name.replace(' ','_')}.pkl")
    with open(mdl_path, "wb") as f:
        pickle.dump(clf, f)   # the ACTUAL fitted object

    cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{name.replace(' ','_')}.csv")
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_path)

    auc_s = f"{m['AUC']}%" if m["AUC"] is not None else "n/a"
    print(f"  Acc={m['Accuracy']:6.2f}% [{ci['Accuracy']['ci_lo']:.1f}-{ci['Accuracy']['ci_hi']:.1f}]  "
          f"P={m['Precision']:6.2f}%  R={m['Recall']:6.2f}%  F1={m['F1']:6.2f}% "
          f"[{ci['F1']['ci_lo']:.1f}-{ci['F1']['ci_hi']:.1f}]  AUC={auc_s}")
    print(f"  Saved -> {out_path}")
    print(f"  Saved -> {mdl_path}  (fitted model, verified via clf.predict smoke-test below)")
    print(f"  Saved -> {cm_path}")

    # Smoke-test: confirm the pickled model is actually fitted and usable.
    with open(mdl_path, "rb") as f:
        reloaded = pickle.load(f)
    _ = reloaded.predict(X_test_sc[:1])
    print(f"  Smoke-test passed: reloaded model predicts without error.")

    return result

print("Evaluation utilities ready (fit-and-return, bootstrap CI, per-class, confusion matrix).")


Evaluation utilities ready (fit-and-return, bootstrap CI, per-class, confusion matrix).


## Step 3 — Baseline MLP (256-128-64, early stopping)

In [5]:
from sklearn.neural_network import MLPClassifier

name = "MLP"
print(f"Training {name} (default configuration) ...")
print("AUC methodology: MLPClassifier's native predict_proba (softmax output) used directly. "
      "AUC is One-vs-Rest, macro-averaged.")

clf = MLPClassifier(hidden_layer_sizes=(256, 128, 64), max_iter=500,
                    random_state=GLOBAL_SEED, early_stopping=True, validation_fraction=0.1)

m = evaluate(clf, X_train_sc, X_test_sc, y_train_sm, y_test_raw)   # fitted IN PLACE
result_baseline = save_full_result(name, m, y_test_raw, class_names)

Training MLP (default configuration) ...
AUC methodology: MLPClassifier's native predict_proba (softmax output) used directly. AUC is One-vs-Rest, macro-averaged.
  ⚠ Small test support (<30 samples) -- per-class scores below should not be over-interpreted (comment 7):
     Leucas_aspera                support=  11  P=83.3%  R=90.9%  F1=87.0%
  Acc= 89.94% [88.8-91.0]  P= 90.51%  R= 89.94%  F1= 90.11% [88.9-91.1]  AUC=98.58%
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/result_MLP.json
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/model_MLP.pkl  (fitted model, verified via clf.predict smoke-test below)
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/confusion_matrix_MLP.csv
  Smoke-test passed: reloaded model predicts without error.


## Step 4 — Optuna Hyperparameter Search

**`TUNE_MODE`** controls how Optuna scores candidate configurations:

* `"val"` (default) — fit on the SMOTE-resampled training set, score weighted F1 on the untouched validation split. This is the protocol used by the other ten classifiers and described in the manuscript.
* `"cv"` — mean weighted F1 of 3-fold stratified CV on the SMOTE-resampled training set (the protocol of the previous run). SMOTE samples are interpolated from real neighbours, so these folds are not independent and the CV score is optimistic.

The test split is never used for tuning in either mode.

In [6]:
import time, optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import StratifiedKFold, cross_val_score

TUNE_MODE = "val"      # "val" (recommended, matches other classifiers) or "cv" (previous protocol)
N_TRIALS  = 40
optuna.logging.set_verbosity(optuna.logging.WARNING)

def build_mlp(p):
    hidden = (p["layer_0"],) if p["n_layers"] == 1 else (p["layer_0"], p["layer_1"])
    return MLPClassifier(hidden_layer_sizes=hidden, alpha=p["alpha"],
                         learning_rate_init=p["lr"], batch_size=p["batch"],
                         max_iter=300, random_state=GLOBAL_SEED, early_stopping=True)

def objective_mlp(trial):
    p = {"n_layers": trial.suggest_int("n_layers", 1, 2),
         "layer_0":  trial.suggest_int("layer_0", 32, 1024)}
    if p["n_layers"] == 2:
        p["layer_1"] = trial.suggest_int("layer_1", 32, 512)
    p["alpha"] = trial.suggest_float("alpha", 1e-5, 1e-1, log=True)
    p["lr"]    = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    p["batch"] = trial.suggest_categorical("batch", [32, 64, 128, 256])
    clf = build_mlp(p)
    if TUNE_MODE == "val":
        clf.fit(X_train_sc, y_train_sm)
        return f1_score(y_val_raw, clf.predict(X_val_sc), average="weighted")
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=GLOBAL_SEED)
    return cross_val_score(clf, X_train_sc, y_train_sm, cv=skf,
                           scoring="f1_weighted", n_jobs=-1).mean()

t_start = time.time()
def progress(study, trial):
    p = trial.params
    arch = (p["layer_0"],) if p["n_layers"] == 1 else (p["layer_0"], p.get("layer_1"))
    print(f"[Trial {trial.number+1:2d}/{N_TRIALS}] F1={trial.value*100:6.2f}%  "
          f"best={study.best_value*100:6.2f}% (trial {study.best_trial.number+1})  "
          f"hidden={arch} alpha={p['alpha']:.1e} lr={p['lr']:.1e} batch={p['batch']}  "
          f"elapsed={(time.time()-t_start)/60:5.1f} min", flush=True)

print(f"Optuna search space: n_layers in {{1,2}}, layer_0 ~ Int[32,1024], layer_1 ~ Int[32,512], "
      f"alpha ~ LogUniform[1e-5,1e-1], lr ~ LogUniform[1e-5,1e-2], batch in {{32,64,128,256}}")
print(f"seed={GLOBAL_SEED}, n_trials={N_TRIALS}, sampler=TPESampler, TUNE_MODE='{TUNE_MODE}'")
print("Objective: " + ("fit on SMOTE-resampled TRAIN, score weighted-F1 on untouched VAL."
                       if TUNE_MODE == "val" else
                       "3-fold stratified CV weighted-F1 on SMOTE-resampled TRAIN (optimistic)."))

study_mlp = optuna.create_study(direction="maximize", sampler=TPESampler(seed=GLOBAL_SEED))
study_mlp.optimize(objective_mlp, n_trials=N_TRIALS, callbacks=[progress])

print(f"\nDone in {(time.time()-t_start)/60:.1f} min")
print(f"Best {TUNE_MODE.upper()} F1(weighted) = {study_mlp.best_value*100:.2f}%  "
      f"(trial {study_mlp.best_trial.number+1})")
print(f"Best params: {study_mlp.best_params}")

with open(os.path.join(OUTPUT_DIR, "optuna_result_MLP.json"), "w") as f:
    json.dump({"Classifier": "MLP_Optuna_BestTrial", "TuneMode": TUNE_MODE,
               "Best_F1_weighted": round(study_mlp.best_value*100, 2),
               "BestParams": study_mlp.best_params, "NumTrials": N_TRIALS,
               "SearchTime_min": round((time.time()-t_start)/60, 1)}, f, indent=2)
print(f"Saved -> {os.path.join(OUTPUT_DIR, 'optuna_result_MLP.json')}")

Optuna search space: n_layers in {1,2}, layer_0 ~ Int[32,1024], layer_1 ~ Int[32,512], alpha ~ LogUniform[1e-5,1e-1], lr ~ LogUniform[1e-5,1e-2], batch in {32,64,128,256}
seed=42, n_trials=40, sampler=TPESampler, TUNE_MODE='val'
Objective: fit on SMOTE-resampled TRAIN, score weighted-F1 on untouched VAL.
[Trial  1/40] F1= 91.67%  best= 91.67% (trial 1)  hidden=(976,) alpha=8.5e-03 lr=6.3e-04 batch=256  elapsed=  6.9 min
[Trial  2/40] F1= 90.11%  best= 91.67% (trial 1)  hidden=(735, 41) alpha=7.6e-02 lr=3.1e-03 batch=256  elapsed= 15.2 min
[Trial  3/40] F1= 91.43%  best= 91.67% (trial 1)  hidden=(460, 172) alpha=2.8e-03 lr=2.6e-05 batch=256  elapsed= 50.1 min
[Trial  4/40] F1= 91.48%  best= 91.67% (trial 1)  hidden=(542,) alpha=2.3e-03 lr=1.4e-05 batch=256  elapsed= 83.0 min
[Trial  5/40] F1= 90.73%  best= 91.67% (trial 1)  hidden=(834, 178) alpha=2.5e-05 lr=1.1e-03 batch=128  elapsed=103.0 min
[Trial  6/40] F1= 91.02%  best= 91.67% (trial 1)  hidden=(288, 350) alpha=1.8e-04 lr=3.6e-04 

## Step 5 — Final Model: Retrain with Best Hyperparameters, Evaluate Once on Test

In [7]:
name = "MLP_Optuna"
print(f"Training {name} (final retrained model, evaluated ONCE on held-out test) ...")

# Read the parameters from the saved search (no hard-coding), so this cell can
# also be run alone after a kernel restart without repeating the search.
with open(os.path.join(OUTPUT_DIR, "optuna_result_MLP.json")) as f:
    opt = json.load(f)
bp = opt["BestParams"]
print(f"Using params from {opt.get('TuneMode', 'cv').upper()}-based search: {bp}")

clf_final = build_mlp(bp)
m_final = evaluate(clf_final, X_train_sc, X_test_sc, y_train_sm, y_test_raw)   # fitted IN PLACE
result_optuna = save_full_result(name, m_final, y_test_raw, class_names)

print("\n" + "="*70)
print("BASELINE (256-128-64, default) vs. OPTUNA-TUNED -- compare on TEST:")
print("="*70)
for lbl, r in [("Baseline", result_baseline), ("Optuna  ", result_optuna)]:
    ci = r["Bootstrap_CI_95"]
    print(f"  {lbl}: Acc={r['Accuracy']:.2f}% [{ci['Accuracy']['ci_lo']:.1f}-{ci['Accuracy']['ci_hi']:.1f}]  "
          f"F1={r['F1']:.2f}% [{ci['F1']['ci_lo']:.1f}-{ci['F1']['ci_hi']:.1f}]  AUC={r['AUC']}%")

Training MLP_Optuna (final retrained model, evaluated ONCE on held-out test) ...
Using params from VAL-based search: {'n_layers': 1, 'layer_0': 916, 'alpha': 8.32247350862405e-05, 'lr': 1.0968703180283828e-05, 'batch': 128}
  ⚠ Small test support (<30 samples) -- per-class scores below should not be over-interpreted (comment 7):
     Leucas_aspera                support=  11  P=100.0%  R=100.0%  F1=100.0%
  Acc= 91.80% [90.6-92.8]  P= 91.99%  R= 91.80%  F1= 91.88% [90.7-92.9]  AUC=99.13%
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/result_MLP_Optuna.json
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/model_MLP_Optuna.pkl  (fitted model, verified via clf.predict smoke-test below)
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/confusion_matrix_MLP_Optuna.csv
  Smoke-test passed: reloaded model predicts without error.

BASELINE (256-128-64, default) vs. OPTUNA-TUNED -- compare on TEST:
  Baseline: Acc=89.94% [88.8-91.0]  F1=90.11% [88.9-91.1]  AUC=98.